## Week 2: Vector Search — Part 2: Using `minsearch` VectorSearch

Instead of manually computing dot products, we use `minsearch.VectorSearch`, which wraps the same idea in a searchable index and supports keyword filtering.

### Setup: load model and data

In [2]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
openai_client = OpenAI()

In [3]:
from ingest import load_faq_data, build_index
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")

documents = load_faq_data()
index = build_index(documents)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [4]:
from tqdm.auto import tqdm
import numpy as np

texts = [doc["question"] + " " + doc["answer"] for doc in documents]

vectors = []
batch_size = 50

for i in tqdm(range(0, len(texts), batch_size)):
    batch = texts[i : i + batch_size]
    batch_vectors = model.encode(batch)
    vectors.extend(batch_vectors)

X = np.array(vectors)
print(f"Embeddings matrix shape: {X.shape}")

  0%|          | 0/28 [00:00<?, ?it/s]

Embeddings matrix shape: (1368, 384)


### Build the minsearch VectorSearch index

In [5]:
from minsearch import VectorSearch

vindex = VectorSearch(keyword_fields=["course"])
vindex.fit(X, documents)

In [6]:
from rag_helper import RAGBase

assistant = RAGBase(index=index, llm_client=openai_client)
assistant.rag("when does the course start")

'You can start whenever you want. The videos and GitHub materials are available, and the deadlines are listed in the course management platform.'

In [ ]:
class RAGVector(RAGBase):
    def __init__(self, embedder, **kwargs):
        super().__init__(**kwargs)
        self.embedder = embedder

    def search(self, query, num_results=5):
        query_vector = self.embedder.encode(query)
        filter_dict = {"course": self.course}

        return self.index.search(
            query_vector, num_results=num_results, filter_dict=filter_dict
        )

In [ ]:
vector_assistant = RAGVector(
    embedder=model,
    index=vindex,
    llm_client=openai_client,
)

vector_assistant.rag("the program has already begun, can I still sign up?")

'Yes — you can still join and start learning. If you want a certificate, make sure you submit your project while submissions are still being accepted.'